In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

print("Libraries imported successfully")

Libraries imported successfully


In [ ]:
df= pd.read_csv("Telco_Customer_Churn_Dataset (cleaned).csv")

print(df.shape)

(7043, 21)


In [ ]:
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 36, float("inf")],
    labels=["New", "Medium-term", "Long-term"]
)

In [ ]:
df[["customer_id", "tenure", "tenure_group"]].head(15)

,customer_id,tenure,tenure_group
0,7590-VHVEG,1,New
1,5575-GNVDE,34,Medium-term
2,3668-QPYBK,2,New
3,7795-CFOCW,45,Long-term
4,9237-HQITU,2,New
5,9305-CDSKC,8,New
6,1452-KIOVK,22,Medium-term
7,6713-OKOMC,10,New
8,7892-POOKP,28,Medium-term
9,6388-TABGU,62,Long-term


In [ ]:
df["tenure_group"].value_counts(sort=False)

tenure_group
New            2186
Medium-term    1856
Long-term      3001
Name: count, dtype: int64

In [ ]:
segment_percentage = (
    df["tenure_group"].value_counts(normalize=True, sort=False) * 100
)

print(segment_percentage.round(2))

tenure_group
New            31.04
Medium-term    26.35
Long-term      42.61
Name: proportion, dtype: float64


In [ ]:
segment_counts = df["tenure_group"].value_counts(sort=False)

print(segment_counts)

tenure_group
New            2186
Medium-term    1856
Long-term      3001
Name: count, dtype: int64


In [ ]:
fig = px.pie(
    names=segment_counts.index,
    values=segment_counts.values,
    title="Customer Distribution by Tenure Group",
    hole=0.45
)

fig.update_traces(
    textposition="inside",
    textinfo="label+percent"
)

fig.update_layout(
    title_x=0.5
)

fig.show()


### Customer Distribution by Tenure Group

Customers were segmented into three categories based on their tenure.

**Key Findings:**

- New customers: 2,186 (31.04%)
- Medium-term customers: 1,856 (26.35%)
- Long-term customers: 3,001 (42.61%)

**Business Observations:**

- Long-term customers represent the largest segment.
- Medium-term customers represent the smallest segment.
- Approximately 31% of customers are relatively new.

**Conclusion:**

The company has a substantial base of long-term customers. However, the new and medium-term segments also represent a significant proportion of its customer base.


In [ ]:
avg_charges = df.groupby(
    ["tenure_group", "churn"],
    observed=False
)["monthly_charges"].mean().reset_index()

print(avg_charges.round(2))

  tenure_group churn  monthly_charges
0          New    No            46.71
1          New   Yes            66.49
2  Medium-term    No            57.26
3  Medium-term   Yes            80.70
4    Long-term    No            69.68
5    Long-term   Yes            89.18


In [ ]:
fig = px.bar(
    avg_charges,
    x="tenure_group",
    y="monthly_charges",
    color="churn",
    barmode="group",
    text_auto=".2f",
    title="Average Monthly Charges by Tenure Group and Churn Status",
    labels={
        "tenure_group": "Customer Tenure Group",
        "monthly_charges": "Average Monthly Charges ($)",
        "churn": "Churn Status"
    }
)

fig.update_layout(
    title_x=0.5,
    legend_title="Churn Status"
)

fig.show()

In [ ]:
comparison = avg_charges.pivot(
    index="tenure_group",
    columns="churn",
    values="monthly_charges"
)

comparison["difference"] = (
    comparison["Yes"] - comparison["No"]
)

comparison.round(2)

churn,No,Yes,difference
tenure_group,,,
New,46.71,66.49,19.78
Medium-term,57.26,80.70,23.43
Long-term,69.68,89.18,19.50


In [ ]:
largest_gap_group = comparison["difference"].abs().idxmax()

largest_gap = comparison.loc[
    largest_gap_group, "difference"
]

print("Largest difference:", largest_gap_group)
print("Difference in monthly charges:", round(largest_gap, 2))

Largest difference: Medium-term
Difference in monthly charges: 23.43


In [ ]:
fig = px.bar(
    avg_charges,
    x="tenure_group",
    y="monthly_charges",
    color="churn",
    barmode="group",
    text_auto=".2f",
    title="Average Monthly Charges by Tenure Group and Churn Status",
    labels={
        "tenure_group": "Customer Tenure Group",
        "monthly_charges": "Average Monthly Charges ($)",
        "churn": "Churn Status"
    }
)


group_data = avg_charges[
    avg_charges["tenure_group"] == largest_gap_group
]

highest_value = group_data["monthly_charges"].max()

fig.add_annotation(
    x=largest_gap_group,
    y=highest_value,
    text=f"Largest gap: ${abs(largest_gap):.2f}",
    showarrow=True,
    arrowhead=2,
    ax=50,
    ay=-50
)

fig.update_layout(title_x=0.5)

fig.show()


## Key Findings – Customer Segmentation Analysis

### 1. Customer Distribution by Tenure

- New customers (0–12 months): 2,186 (31.04%).
- Medium-term customers (13–36 months): 1,856 (26.35%).
- Long-term customers (37+ months): 3,001 (42.61%).
- Long-term customers represent the largest segment of the customer base.

### 2. Average Monthly Charges by Tenure and Churn Status

- New customers:
  - Retained customers: $46.71
  - Churned customers: $66.49

- Medium-term customers:
  - Retained customers: $57.26
  - Churned customers: $80.70

- Long-term customers:
  - Retained customers: $69.68
  - Churned customers: $89.18

### 3. Significant Trends

- Churned customers have higher average monthly charges than retained customers across all three tenure groups.
- Medium-term customers show the largest difference in average monthly charges, approximately $23.43.
- Average monthly charges increase with tenure for both churned and retained customers.
- Long-term customers have the highest average monthly charges in both churn categories.

### 4. Business Insights

- The company should investigate the relationship between monthly pricing and customer churn.
- Medium-term customers deserve further attention because they show the largest difference in average monthly charges.
- Targeted retention strategies and personalised offers could be explored to improve customer retention.

### Conclusion

The segmentation analysis highlights differences in customer distribution and monthly spending behaviour across tenure groups. Churned customers consistently show higher average monthly charges than retained customers. These findings provide useful insights for further investigation into customer churn and retention strategies.
